In [1]:
from pyrocko import util, model, io, trace, moment_tensor, gmtpy,orthodrome
# from seiscloud import plot as scp
# from seiscloud import cluster as scc
import numpy as np
import os
# import shutil
import matplotlib.pyplot as plt

import re
from pathlib import Path
from datetime import datetime

import yaml

# CLASS TO LOAD .YAML FILES AND READ PARAMENTER RESULTS
class IgnoreTagsLoader(yaml.SafeLoader):
    pass

def ignore_unknown(loader, tag_suffix, node):
    if isinstance(node, yaml.MappingNode):
        return loader.construct_mapping(node)
    elif isinstance(node, yaml.SequenceNode):
        return loader.construct_sequence(node)
    else:
        return loader.construct_scalar(node)

IgnoreTagsLoader.add_multi_constructor('!', ignore_unknown)

In [2]:
workdir='../'

catdir=os.path.join(workdir,'CAT')
catname=os.path.join(catdir,'catalogue_flegrei_VLP.pf')                 # CHANGE
refevents=model.load_events(catname)
mttargets = [ev for ev in refevents]

badmtsols = ['']    # exclude some events
print(f'Catalogue: {catname}')
print('All events in catalogue:', len(mttargets))
goodmttargets = [ev for ev in mttargets if ev.name not in badmtsols]
print('Good events in catalogue:', len(goodmttargets))

reference_config_file = 'flegrei.2023.06.11.06.44.25.composite.LF.std.gronf' # CHANGE: config file da cui partire
config_path = os.path.join(workdir, 'CONFIG')
config_name = os.path.join(config_path, reference_config_file)       


# Insert source prefixes 
par_names = ['time','north_shift','east_shift',
              'depth','magnitude',
              'rmnn','rmee','rmdd','rmne','rmnd','rmed',
            ]
par_stats_names = ['best','percentile16','percentile84']
ev_stats={}

Catalogue: ../CAT/catalogue_flegrei_VLP.pf
All events in catalogue: 22
Good events in catalogue: 22


In [3]:
#reportdir=os.path.join(workdir,'report') # pre-defined
reportdir = '/Users/giaco/UNI/PhD_CODE/GIT/CAMPI_FLEGREI_moment_tensor/report'

counter_reports = 0
for ev in goodmttargets:
    counter = 0
    for vrs in ['cmt_devi_XXL_final_','cmt_devi_XL_final_', 'cmt_devi_L_final_', 
                'cmt_devi_M_final_','cmt_devi_S_final_','cmt_devi_S_']: # main report
        targetdir = os.path.join(reportdir, ev.name, vrs + ev.name)
        #if not os.path.isdir(targetdir):
            #print(ev.name, 'missing report dir', targetdir)
        if os.path.isdir(targetdir):
            counter += 1
            fname = os.path.join(targetdir, 'stats.yaml')     # results
            if os.path.isfile(fname):
                # Read file
                with open(fname, 'r') as f:
                    data = yaml.load(f, Loader=IgnoreTagsLoader)    # unsing loader class
                
                # acces 'paramenter_stats_list'
                parameter_stats = data['parameter_stats_list']
                # print(parameter_stats)
                ev_stats[ev.name]={par:0 for par in par_names}
                for p in parameter_stats:
                    if p['name'] in par_names:
                        ev_stats[ev.name][p['name']]= {'best':f"{p['best']:.8g}",
                                                       'percentile16': f"{p['percentile16']:.8g}",
                                                       'percentile84': f"{p['percentile84']:.8g}"}
                        # print values
                        #print(p['name'], p['best'], p['percentile16'], p['percentile84'])

    if counter == 0:
        print(f'WARNING: {ev.name} does not have any report directories!')
    elif counter > 1:
        print(f'WARNING: {ev.name} has multiple report directories!')
    else:
        print(f'NICE! {ev.name} has report directory: {targetdir}')
        counter_reports += 1

print(f'Number of events with report directories: {counter_reports} / {len(goodmttargets)}')

NICE! flegrei_2018_09_18_21_36_41 has report directory: /Users/giaco/UNI/PhD_CODE/GIT/CAMPI_FLEGREI_moment_tensor/report/flegrei_2018_09_18_21_36_41/cmt_devi_S_flegrei_2018_09_18_21_36_41
NICE! flegrei_2023_06_11_06_44_25 has report directory: /Users/giaco/UNI/PhD_CODE/GIT/CAMPI_FLEGREI_moment_tensor/report/flegrei_2023_06_11_06_44_25/cmt_devi_S_flegrei_2023_06_11_06_44_25
NICE! flegrei_2023_09_07_17_45_28 has report directory: /Users/giaco/UNI/PhD_CODE/GIT/CAMPI_FLEGREI_moment_tensor/report/flegrei_2023_09_07_17_45_28/cmt_devi_S_flegrei_2023_09_07_17_45_28
NICE! flegrei_2023_09_26_07_10_29 has report directory: /Users/giaco/UNI/PhD_CODE/GIT/CAMPI_FLEGREI_moment_tensor/report/flegrei_2023_09_26_07_10_29/cmt_devi_S_flegrei_2023_09_26_07_10_29
NICE! flegrei_2023_10_02_20_08_26 has report directory: /Users/giaco/UNI/PhD_CODE/GIT/CAMPI_FLEGREI_moment_tensor/report/flegrei_2023_10_02_20_08_26/cmt_devi_S_flegrei_2023_10_02_20_08_26
NICE! flegrei_2024_04_27_03_44_56 has report directory: /Use

In [4]:
print('test ev_stats:',ev_stats['flegrei_2024_05_22_06_28_00']['magnitude']['percentile16'],
      '-->',ev_stats['flegrei_2024_05_22_06_28_00']['magnitude']['percentile84'])

test ev_stats: 3.095994 --> 3.4457273


In [5]:
# Create list w/ new values for new config files
variants=[]
source_name='vt' # CHANGE: FROM CONFIG FILE

for ev_name in ev_stats:
    var_dict = {}
    var_dict['datetime'] = datetime.strptime(ev_name, 'flegrei_%Y_%m_%d_%H_%M_%S')
    var_dict['name'] = source_name
    var_dict['ranges'] = {}
    for par_name in ev_stats[ev_name]:
        if par_name == 'time':
            var_dict['ranges'][par_name] = f"'{ev_stats[ev_name][par_name]['percentile16']} .. {ev_stats[ev_name][par_name]['percentile84']} | add'"
        else:
            var_dict['ranges'][par_name] = f"{ev_stats[ev_name][par_name]['percentile16']} .. {ev_stats[ev_name][par_name]['percentile84']}"
    variants.append(var_dict)

print('test variants:',variants[1])

test variants: {'datetime': datetime.datetime(2023, 6, 11, 6, 44, 25), 'name': 'vt', 'ranges': {'time': "'-0.23873581 .. 0.39286542 | add'", 'north_shift': '-321.41894 .. 1708.3156', 'east_shift': '-1553.3043 .. 326.37168', 'depth': '2812.9988 .. 4311.5466', 'magnitude': '2.7747531 .. 3.1006486', 'rmnn': '0.32730309 .. 0.89255625', 'rmee': '-0.35405119 .. 0.2918408', 'rmdd': '-0.89578288 .. -0.29230385', 'rmne': '-0.3410317 .. 0.40010144', 'rmnd': '-0.44809926 .. 0.44916153', 'rmed': '-0.037526627 .. 0.61144367'}}


In [6]:
# classe per creare nuovi config files

def find_subproblem_block(text: str, name: str) -> tuple[int, int]:
    block_pattern = re.compile(
        r'^(?P<indent>[ \t]*)- !grond\.CMTSubProblemConfig[ \t]*\n'
        r'(?:[ \t]+\S[^\n]*\n)*?'
        r'(?P=indent)[ \t]+name:[ \t]*' + re.escape(name) + r'[ \t]*$',
        re.MULTILINE
    )
    m = block_pattern.search(text)
    if m is None:
        raise ValueError(f"Blocco CMTSubProblemConfig con name='{name}' non trovato")

    block_start = m.start()
    indent_len = len(m.group('indent'))
    end_pattern = re.compile(r'^[ \t]{0,' + str(indent_len) + r'}\S', re.MULTILINE)
    end_match = end_pattern.search(text, m.end())
    block_end = end_match.start() if end_match else len(text)
    return block_start, block_end


def make_dst_path(src_path: Path, src_date_pattern: re.Pattern, dt: datetime) -> Path:
    new_date = dt.strftime('%Y.%m.%d.%H.%M.%S')
    new_name = src_date_pattern.sub(new_date, src_path.name, count=1)
    return src_path.with_name(new_name)


def generate_variants(src_path: str | Path, variants: list[dict]) -> None:
    """
    Per ogni variante in `variants`, crea una copia di src_path con:
      - nome aggiornato con la data fornita
      - ranges aggiornati nel blocco CMTSubProblemConfig indicato

    Ogni elemento di `variants` è un dict con:
      - 'datetime': datetime object con la nuova data/ora
      - 'name':     str, nome del blocco da modificare ('vt' o 'vlp')
      - 'ranges':   dict { 'param': 'nuovo_valore' }

    Esempio:
      variants = [
          {
              'datetime': datetime(2023, 7, 1, 8, 30, 0),
              'name': 'vt',
              'ranges': {
                  'depth':     '3000 .. 4500',
                  'magnitude': '2.80 .. 3.20',
                  'time':      "'-0.30 .. 0.50 | add'",
              },
          },
          ...
      ]
    """
    src_path = Path(src_path)
    base_text = src_path.read_text()

    # estrae la data del file sorgente una volta sola, usata come pattern fisso
    src_date_match = re.search(r'\d{4}\.\d{2}\.\d{2}\.\d{2}\.\d{2}\.\d{2}', src_path.name)
    if src_date_match is None:
        raise ValueError(f"Nessuna data trovata nel nome sorgente '{src_path.name}'")
    src_date_pattern = re.compile(re.escape(src_date_match.group()))

    for i, v in enumerate(variants):
        dt     = v['datetime']
        name   = v['name']
        ranges = v['ranges']

        # aggiorna il blocco corretto
        text = base_text
        start, end = find_subproblem_block(text, name)
        block = text[start:end]

        for param, value in ranges.items():
            pattern = rf'^([ \t]+{re.escape(param)}:[ \t]).*$'
            new_block, n = re.subn(pattern, rf'\g<1>{value}', block, count=1, flags=re.MULTILINE)
            if n == 0:
                print(f"  ⚠️  [{i+1}] parametro '{param}' non trovato nel blocco '{name}'")
            else:
                block = new_block

        text = text[:start] + block + text[end:]

        # costruisce il path di destinazione con la nuova data nel nome
        dst_path = make_dst_path(src_path, src_date_pattern, dt)

        if dst_path.exists():
            print(f"  ⚠️  [{i+1}] file già esistente, verrà sovrascritto: {dst_path.name}")
        dst_path.write_text(text)
        print(f"  ✅ [{i+1}] salvato: {dst_path.name}")


# ---------------------------------------------------------------------------

generate_variants(
    src_path=config_name,
    variants=variants,
)

  ⚠️  [1] file già esistente, verrà sovrascritto: flegrei.2018.09.18.21.36.41.composite.LF.std.gronf
  ✅ [1] salvato: flegrei.2018.09.18.21.36.41.composite.LF.std.gronf
  ⚠️  [2] file già esistente, verrà sovrascritto: flegrei.2023.06.11.06.44.25.composite.LF.std.gronf
  ✅ [2] salvato: flegrei.2023.06.11.06.44.25.composite.LF.std.gronf
  ⚠️  [3] file già esistente, verrà sovrascritto: flegrei.2023.09.07.17.45.28.composite.LF.std.gronf
  ✅ [3] salvato: flegrei.2023.09.07.17.45.28.composite.LF.std.gronf
  ⚠️  [4] file già esistente, verrà sovrascritto: flegrei.2023.09.26.07.10.29.composite.LF.std.gronf
  ✅ [4] salvato: flegrei.2023.09.26.07.10.29.composite.LF.std.gronf
  ⚠️  [5] file già esistente, verrà sovrascritto: flegrei.2023.10.02.20.08.26.composite.LF.std.gronf
  ✅ [5] salvato: flegrei.2023.10.02.20.08.26.composite.LF.std.gronf
  ⚠️  [6] file già esistente, verrà sovrascritto: flegrei.2024.04.27.03.44.56.composite.LF.std.gronf
  ✅ [6] salvato: flegrei.2024.04.27.03.44.56.composite